# ChronoPDE V2 Phase 4 — matched feasibility training

Attach the private Phase 3 dataset containing `chronopde_v2_development.h5`, enable Internet, and select one T4 GPU. This notebook never reads or generates confirmatory states. The second GPU in a T4×2 session is intentionally unused because the frozen comparison is single-device.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
BRANCH = 'codex/chronopde-v2-phase4'
REPOSITORY = Path('/kaggle/working/ChronoPDE')
if not REPOSITORY.exists():
    clone = ['git', 'clone', '--branch', BRANCH, '--single-branch']
    subprocess.run([*clone, REPOSITORY_URL, str(REPOSITORY)], check=True)
install = [sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPOSITORY)]
subprocess.run(install, check=True)
commit = subprocess.check_output(
    ['git', '-C', str(REPOSITORY), 'rev-parse', 'HEAD'], text=True
).strip()
print('Commit:', commit)
print('PyTorch:', __import__('torch').__version__)
print('GPU:', __import__('torch').cuda.get_device_name(0))

In [ ]:
candidates = list(Path('/kaggle/input').rglob('chronopde_v2_development.h5'))
assert len(candidates) == 1, f'Expected exactly one Phase 3 HDF5, found: {candidates}'
DATA = candidates[0]
print('Dataset:', DATA, f'({DATA.stat().st_size / 2**30:.2f} GiB)')
check_command = [
    sys.executable, 'scripts/chronopde_v2.py', 'phase4',
    '--data-path', str(DATA), '--check-only', '--format', 'json',
]
subprocess.run(check_command, cwd=REPOSITORY, check=True)

In [ ]:
RESUME = False  # Set True only when last.pt and the running manifest were restored.
return_codes = {}
for model in ('fft', 'dct'):
    command = [
        sys.executable, 'scripts/chronopde_v2.py', 'phase4',
        '--data-path', str(DATA), '--model', model,
        '--device', 'cuda', '--format', 'json',
    ]
    if RESUME:
        command.append('--resume')
    print('Running', model, flush=True)
    return_codes[model] = subprocess.run(command, cwd=REPOSITORY).returncode

archive_base = Path('/kaggle/working/chronopde_v2_phase4_outputs')
staging = Path('/kaggle/working/phase4_package')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()
for relative in ('artifacts/chronopde_v2/runs', 'reports/chronopde_v2/phase4'):
    source = REPOSITORY / relative
    if source.exists():
        destination = staging / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination, dirs_exist_ok=True)
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=staging)
print('Return codes:', return_codes)
print('Download:', archive)
message = 'A run failed; download the ZIP before debugging.'
assert all(code == 0 for code in return_codes.values()), message